# START TRAINING

In [1]:
import ast
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

warnings.simplefilter(action="ignore", category=UserWarning)

In [2]:
final = (
    pd.read_csv('data_2026/final_features.csv')
    .rename(columns={
        'WTEAMID': 'W_TEAMID',
        'LTEAMID': 'L_TEAMID',
        'WSCORE': 'W_SCORE',
        'LSCORE': 'L_SCORE',
    })
)
# W_TEAMID is always the winner in historical tourney data
final['WIN_INDICATOR'] = 1

/var/folders/fn/v37r1g5j74928hgvsvhgtlhh0000gn/T/ipykernel_52559/1924972115.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final['WIN_INDICATOR'] = 1


In [3]:
# Columns with null values and their respective counts
{col: int(final[col].isna().sum()) for col in final.columns if final[col].isna().any()}

{}

In [4]:
drop_cols = ['W_CTWINS', 'W_AVERAGECTSCORE', 'L_CTWINS', 'L_AVERAGECTSCORE']
final = final.drop(columns=[c for c in drop_cols if c in final.columns])
final = final.drop(columns=[c for c in ['W_WLOCN','W_WLOCH','W_WLOCA','L_WLOCN','L_WLOCH','L_WLOCA'] if c in final.columns])  # variants

In [5]:
parameters = {
    "n_estimators": [100, 200, 300, 400, 500],
    "learning_rate": [0.1, 0.2, 0.3, 0.4, 0.5],
    "max_depth": list(range(3, 6, 1)),
    "min_child_weight": list(range(1, 6, 1)),
}

In [6]:
NON_FEATURE_COLS = {'SEASON', 'WIN_INDICATOR', 'L_TEAMID', 'W_TEAMID', 'W_SCORE', 'L_SCORE', 'ROUND', 'L_REGION', 'W_REGION'}
feature_cols = [c for c in final.columns if c not in NON_FEATURE_COLS]

train = final[(final.SEASON >= 2010) & (final.SEASON <= 2023)].copy()
test = final[final.SEASON > 2023].copy()

BRACKET_SEASON = 2026  # Season to predict the bracket for

### Swap the W and L teams

In [7]:
# Double training data with both W/L orientations — perfectly balanced, reproducible, 2x data
df = train.copy()
w_cols = sorted([c for c in df.columns if c.startswith('W_')])
l_cols = ['L_' + c[2:] for c in w_cols]

df_swapped = df.copy()
for w, l in zip(w_cols, l_cols):
    df_swapped[w] = df[l]
    df_swapped[l] = df[w]

train = pd.concat([df, df_swapped], ignore_index=True)
train['WIN_INDICATOR'] = (train['W_SCORE'] > train['L_SCORE']).astype(int)

In [8]:
X_train = train[feature_cols]
y_train = train['WIN_INDICATOR']

all_rounds = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss'),
    param_grid=parameters,
    n_jobs=-1,
    scoring="accuracy",
    cv=5,
)
all_rounds.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 0.2, ...], 'max_depth': [3, 4, ...], 'min_child_weight': [1, 2, ...], 'n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold an

In [9]:
def swap_wl(df):
    """Return a copy of df with all W_ and L_ columns swapped, preserving dtypes."""
    out = df.copy()
    w_cols = sorted([c for c in df.columns if c.startswith('W_')])
    l_cols = ['L_' + c[2:] for c in w_cols]
    for w, l in zip(w_cols, l_cols):
        out[w] = df[l]
        out[l] = df[w]
    return out

# Unbiased evaluation: each game tested in both orientations, WIN_INDICATOR derived from score
test_r1 = test[test.ROUND == 1].copy()
test_eval = pd.concat([test_r1, swap_wl(test_r1)], ignore_index=True)
test_eval['WIN_INDICATOR'] = (test_eval['W_SCORE'] > test_eval['L_SCORE']).astype(int)

test_eval['PRED_WIN_INDICATOR'] = all_rounds.predict(test_eval[feature_cols])

for season in sorted(test_eval.SEASON.unique()):
    acc = accuracy_score(
        test_eval[test_eval.SEASON == season]['WIN_INDICATOR'],
        test_eval[test_eval.SEASON == season]['PRED_WIN_INDICATOR'],
    )
    print(f'Accuracy {season}: {acc:.4f}')
    globals()[f'accuracy_{season}'] = acc

accuracy = accuracy_score(test_eval['WIN_INDICATOR'], test_eval['PRED_WIN_INDICATOR'])
print(f'Accuracy total: {accuracy:.4f}')

Accuracy 2024: 0.7188
Accuracy 2025: 0.7969
Accuracy total: 0.7578


/var/folders/fn/v37r1g5j74928hgvsvhgtlhh0000gn/T/ipykernel_52559/4249703926.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_eval['PRED_WIN_INDICATOR'] = all_rounds.predict(test_eval[feature_cols])


#### Save the trained model

In [10]:
import os
os.makedirs('model', exist_ok=True)

optimal_model = all_rounds.best_estimator_
joblib.dump(optimal_model, 'model/march_madness_model.joblib')
print(f'Model saved. Best params: {all_rounds.best_params_}')
print(f'Test seasons: {sorted(test_eval.SEASON.unique())}')

Model saved. Best params: {'learning_rate': 0.1, 'max_depth': 4, 'min_child_weight': 4, 'n_estimators': 100}
Test seasons: [np.int64(2024), np.int64(2025)]


import os
os.makedirs('model', exist_ok=True)

optimal_model = all_rounds.best_estimator_
joblib.dump(optimal_model, 'model/march_madness_model.joblib')
print(f'Model saved. Best params: {all_rounds.best_params_}')
print(f'Test seasons: {sorted(test_eval.SEASON.unique())}')

In [11]:
def predict_games(model, df, feature_cols):
    """Run model predictions on a games dataframe, filling missing feature cols with 0."""
    df = df.copy()
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0
    df['PRED_WIN_INDICATOR'] = model.predict(df[feature_cols])
    return df


def add_team_names(result, teams):
    """Join team names onto a result dataframe that has W_TEAMID and L_TEAMID."""
    result = result.merge(
        teams[['TEAMID', 'TEAMNAME']], left_on='L_TEAMID', right_on='TEAMID'
    ).rename(columns={'TEAMNAME': 'L_TEAM_NAME'}).drop(columns=['TEAMID'])
    result = result.merge(
        teams[['TEAMID', 'TEAMNAME']], left_on='W_TEAMID', right_on='TEAMID'
    ).rename(columns={'TEAMNAME': 'W_TEAM_NAME'}).drop(columns=['TEAMID'])
    return result


def get_round_winners(result):
    """Pick the predicted winner from each game, returning (W_TEAM_ID, W_TEAM_NAME, W_SEED, W_REGION)."""
    r = result.copy()
    r['W_TEAM_ID']   = np.where(r.PRED_WIN_INDICATOR == 1, r.W_TEAMID,    r.L_TEAMID)
    r['W_TEAM_NAME'] = np.where(r.PRED_WIN_INDICATOR == 1, r.W_TEAM_NAME, r.L_TEAM_NAME)
    r['W_SEED']      = np.where(r.PRED_WIN_INDICATOR == 1, r.W_SEED,      r.L_SEED)
    r['W_REGION']    = np.where(r.PRED_WIN_INDICATOR == 1, r.W_REGION,    r.L_REGION)
    return r[['W_TEAM_ID', 'W_TEAM_NAME', 'W_SEED', 'W_REGION']]


def add_season_stats(matchups, season_stats, season_year):
    """Join season stats onto matchups that have WTEAMID2 and LTEAMID2 columns."""
    season = season_stats[season_stats.SEASON == season_year].drop(columns=['SEASON'])
    # Drop REGION if present — it would conflict with W_REGION/L_REGION already in matchups
    season = season.drop(columns=[c for c in ['REGION'] if c in season.columns])
    season_w = season.copy()
    season_w.columns = ['W_' + c for c in season_w.columns]
    season_l = season.copy()
    season_l.columns = ['L_' + c for c in season_l.columns]

    df = matchups.merge(season_w, left_on='WTEAMID2', right_on='W_TEAMID').drop(columns=['W_TEAMID'])
    df = df.merge(season_l, left_on='LTEAMID2', right_on='L_TEAMID').drop(columns=['L_TEAMID'])
    df = df.rename(columns={'WTEAMID2': 'W_TEAMID', 'LTEAMID2': 'L_TEAMID'})
    return df


teams = pd.read_csv('data_2026/MTeams.csv')
teams.columns = teams.columns.str.upper()

season_stats = pd.read_csv('data_2026/final_season_stats.csv')

In [12]:
# Load seeds for the bracket season
seeds_raw = pd.read_csv('data_2026/MNCAATourneySeeds.csv')
seeds_raw.columns = seeds_raw.columns.str.upper()
seeds_bracket = seeds_raw[seeds_raw.SEASON == BRACKET_SEASON].copy()
seeds_bracket['REGION'] = seeds_bracket['SEED'].str[0]
seeds_bracket['IS_PLAYIN'] = seeds_bracket['SEED'].str[1:].str.contains('[ab]', regex=True)
seeds_bracket['SEED_NUM'] = seeds_bracket['SEED'].str[1:].str.replace('[a-z]', '', regex=True).astype(int)

playin_seeds = seeds_bracket[seeds_bracket.IS_PLAYIN][['TEAMID', 'REGION', 'SEED_NUM']].rename(columns={'SEED_NUM': 'SEED'})
regular_seeds = seeds_bracket[~seeds_bracket.IS_PLAYIN][['TEAMID', 'REGION', 'SEED_NUM']].rename(columns={'SEED_NUM': 'SEED'})

# Build play-in matchup pairs (same REGION + SEED = two teams competing for one spot)
playin_rows = []
for (region, seed), group in playin_seeds.groupby(['REGION', 'SEED']):
    team_ids = group.TEAMID.tolist()
    if len(team_ids) == 2:
        playin_rows.append({
            'WTEAMID2': team_ids[0], 'LTEAMID2': team_ids[1],
            'W_SEED': seed, 'L_SEED': seed,
            'W_REGION': region, 'L_REGION': region,
        })

if playin_rows:
    playin_matchups = pd.DataFrame(playin_rows)
    games_playin = add_season_stats(playin_matchups, season_stats, BRACKET_SEASON)
    result_playin = predict_games(optimal_model, games_playin, feature_cols)
    result_playin = add_team_names(result_playin, teams)
    playin_result = result_playin[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR']]
    playin_winners = get_round_winners(playin_result)
    print(f'Play-in winners ({BRACKET_SEASON}):')
    print(playin_winners[['W_TEAM_NAME', 'W_SEED', 'W_REGION']].sort_values('W_REGION').to_string(index=False))
else:
    playin_winners = pd.DataFrame(columns=['W_TEAM_ID', 'W_TEAM_NAME', 'W_SEED', 'W_REGION'])
    print(f'No play-in games found for {BRACKET_SEASON}')

Play-in winners (2026):
W_TEAM_NAME  W_SEED W_REGION
     Lehigh      16        X
        SMU      11        Y
       UMBC      16        Y
   NC State      11        Z


## Round of 64

In [13]:
ROUND_1_PAIRS = [(1, 16), (8, 9), (5, 12), (4, 13), (6, 11), (3, 14), (7, 10), (2, 15)]

# Build Round 1: use regular seeds + play-in winners for their contested spots
round1_seeds = regular_seeds.copy()
if not playin_winners.empty:
    playin_w_seeds = playin_winners.rename(
        columns={'W_TEAM_ID': 'TEAMID', 'W_SEED': 'SEED', 'W_REGION': 'REGION'}
    )[['TEAMID', 'REGION', 'SEED']]
    round1_seeds = pd.concat([round1_seeds, playin_w_seeds], ignore_index=True)

round1_rows = []
for region in round1_seeds.REGION.unique():
    r = round1_seeds[round1_seeds.REGION == region].set_index('SEED')
    for s1, s2 in ROUND_1_PAIRS:
        if s1 in r.index and s2 in r.index:
            round1_rows.append({
                'WTEAMID2': r.loc[s1, 'TEAMID'], 'LTEAMID2': r.loc[s2, 'TEAMID'],
                'W_SEED': s1, 'L_SEED': s2,
                'W_REGION': region, 'L_REGION': region,
            })

round1_matchups = pd.DataFrame(round1_rows)
games_r1 = add_season_stats(round1_matchups, season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, games_r1, feature_cols)
result = add_team_names(result, teams)
print(f'Round of 64 — {BRACKET_SEASON}: {len(result)} games')

Round of 64 — 2026: 32 games


In [14]:
round_1_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR']]
round_1_results.sort_values('W_REGION')

,L_TEAMID,W_TEAMID,L_TEAM_NAME,L_SEED,W_SEED,L_REGION,W_TEAM_NAME,W_REGION,PRED_WIN_INDICATOR
0,1373,1181,Siena,16,1,W,Duke,W,1
1,1395,1326,TCU,9,8,W,Ohio St,W,0
2,1320,1385,Northern Iowa,12,5,W,St John's,W,1
3,1465,1242,Cal Baptist,13,4,W,Kansas,W,1
4,1378,1257,South Florida,11,6,W,Louisville,W,1
5,1295,1277,N Dakota St,14,3,W,Michigan St,W,1
6,1416,1417,UCF,10,7,W,UCLA,W,1
7,1202,1163,Furman,15,2,W,Connecticut,W,1
14,1401,1388,Texas A&M,10,7,X,St Mary's CA,X,1
13,1335,1228,Penn,14,3,X,Illinois,X,1


# Winners round 1

In [15]:
round_1_winners = get_round_winners(round_1_results)
round_1_winners.sort_values('W_REGION')

,W_TEAM_ID,W_TEAM_NAME,W_SEED,W_REGION
0,1181,Duke,1,W
1,1395,TCU,9,W
2,1385,St John's,5,W
3,1242,Kansas,4,W
4,1257,Louisville,6,W
5,1277,Michigan St,3,W
6,1417,UCLA,7,W
7,1163,Connecticut,2,W
14,1388,St Mary's CA,7,X
13,1228,Illinois,3,X


In [16]:
ROUND_2_SEED_PAIRS = [
    (1,8),(1,9),(16,8),(16,9),
    (4,5),(4,12),(13,5),(13,12),
    (3,6),(3,11),(14,6),(14,11),
    (2,7),(2,10),(15,7),(15,10),
]

df1 = round_1_winners.rename(columns={'W_TEAM_NAME':'W_TEAM_NAME','W_REGION':'W_REGION','W_TEAM_ID':'WTEAMID2','W_SEED':'W_SEED'})
df2 = round_1_winners.rename(columns={'W_TEAM_NAME':'L_TEAM_NAME','W_REGION':'L_REGION','W_TEAM_ID':'LTEAMID2','W_SEED':'L_SEED'})

cross = df1.merge(df2, how='cross', suffixes=('','_r'))
seed_set = set(ROUND_2_SEED_PAIRS)
mask = (
    (cross.W_REGION == cross.L_REGION) &
    cross.apply(lambda r: (r.W_SEED, r.L_SEED) in seed_set, axis=1)
)
second_round_matchups = cross[mask][['W_TEAM_NAME','W_REGION','WTEAMID2','W_SEED','L_TEAM_NAME','L_REGION','LTEAMID2','L_SEED']]
second_round_matchups.sort_values('W_REGION')

,W_TEAM_NAME,W_REGION,WTEAMID2,W_SEED,L_TEAM_NAME,L_REGION,LTEAMID2,L_SEED
1,Duke,W,1181,1,TCU,W,1395,9
98,Kansas,W,1242,4,St John's,W,1385,5
164,Michigan St,W,1277,3,Louisville,W,1257,6
230,Connecticut,W,1163,2,UCLA,W,1417,7
265,Florida,X,1196,1,Clemson,X,1155,8
362,Nebraska,X,1304,4,Vanderbilt,X,1435,5
428,Illinois,X,1228,3,North Carolina,X,1314,6
494,Houston,X,1222,2,St Mary's CA,X,1388,7
529,Michigan,Y,1276,1,Georgia,Y,1208,8
626,Alabama,Y,1104,4,Texas Tech,Y,1403,5


# ROUND OF 32!!!

In [17]:
second_round_matchups[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
).sort_values('REGION')

,TEAM_1,TEAM_2,SEED_1,SEED_2,REGION
1,Duke,TCU,1,9,W
98,Kansas,St John's,4,5,W
164,Michigan St,Louisville,3,6,W
230,Connecticut,UCLA,2,7,W
265,Florida,Clemson,1,8,X
362,Nebraska,Vanderbilt,4,5,X
428,Illinois,North Carolina,3,6,X
494,Houston,St Mary's CA,2,7,X
529,Michigan,Georgia,1,8,Y
626,Alabama,Texas Tech,4,5,Y


In [18]:
games_32 = add_season_stats(second_round_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, games_32, feature_cols)
result = add_team_names(result, teams)

round_2_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR']]
round_2_results

,L_TEAMID,W_TEAMID,L_TEAM_NAME,L_SEED,W_SEED,L_REGION,W_TEAM_NAME,W_REGION,PRED_WIN_INDICATOR
0,1395,1181,TCU,9,1,W,Duke,W,1
1,1385,1242,St John's,5,4,W,Kansas,W,1
2,1257,1277,Louisville,6,3,W,Michigan St,W,1
3,1417,1163,UCLA,7,2,W,Connecticut,W,1
4,1155,1196,Clemson,8,1,X,Florida,X,1
5,1435,1304,Vanderbilt,5,4,X,Nebraska,X,1
6,1314,1228,North Carolina,6,3,X,Illinois,X,1
7,1388,1222,St Mary's CA,7,2,X,Houston,X,1
8,1208,1276,Georgia,8,1,Y,Michigan,Y,1
9,1403,1104,Texas Tech,5,4,Y,Alabama,Y,0


In [19]:
round_2_winners = get_round_winners(round_2_results)
round_2_winners.sort_values('W_REGION')

,W_TEAM_ID,W_TEAM_NAME,W_SEED,W_REGION
0,1181,Duke,1,W
1,1242,Kansas,4,W
2,1277,Michigan St,3,W
3,1163,Connecticut,2,W
4,1196,Florida,1,X
5,1304,Nebraska,4,X
6,1228,Illinois,3,X
7,1222,Houston,2,X
8,1276,Michigan,1,Y
9,1403,Texas Tech,5,Y


# Round 2 winners

In [20]:
ROUND_3_SEED_PAIRS = [
    (s1, s2)
    for s1 in [1,16,8,9]
    for s2 in [5,12,4,13]
] + [
    (s1, s2)
    for s1 in [6,11,3,14]
    for s2 in [7,10,2,15]
]

df1 = round_2_winners.rename(columns={'W_TEAM_NAME':'W_TEAM_NAME','W_REGION':'W_REGION','W_TEAM_ID':'WTEAMID2','W_SEED':'W_SEED'})
df2 = round_2_winners.rename(columns={'W_TEAM_NAME':'L_TEAM_NAME','W_REGION':'L_REGION','W_TEAM_ID':'LTEAMID2','W_SEED':'L_SEED'})

cross = df1.merge(df2, how='cross', suffixes=('','_r'))
seed_set = set(ROUND_3_SEED_PAIRS)
mask = (
    (cross.W_REGION == cross.L_REGION) &
    cross.apply(lambda r: (r.W_SEED, r.L_SEED) in seed_set, axis=1)
)
third_round_matchups = cross[mask][['W_TEAM_NAME','W_REGION','WTEAMID2','W_SEED','L_TEAM_NAME','L_REGION','LTEAMID2','L_SEED']]
third_round_matchups.sort_values('W_REGION')

,W_TEAM_NAME,W_REGION,WTEAMID2,W_SEED,L_TEAM_NAME,L_REGION,LTEAMID2,L_SEED
1,Duke,W,1181,1,Kansas,W,1242,4
35,Michigan St,W,1277,3,Connecticut,W,1163,2
69,Florida,X,1196,1,Nebraska,X,1304,4
103,Illinois,X,1228,3,Houston,X,1222,2
137,Michigan,Y,1276,1,Texas Tech,Y,1403,5
171,Virginia,Y,1438,3,Iowa St,Y,1235,2
205,Arizona,Z,1112,1,Arkansas,Z,1116,4
239,Gonzaga,Z,1211,3,Purdue,Z,1345,2


# SWEET SIXTEEN!!!

In [21]:
third_round_matchups[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
).sort_values('REGION')

,TEAM_1,TEAM_2,SEED_1,SEED_2,REGION
1,Duke,Kansas,1,4,W
35,Michigan St,Connecticut,3,2,W
69,Florida,Nebraska,1,4,X
103,Illinois,Houston,3,2,X
137,Michigan,Texas Tech,1,5,Y
171,Virginia,Iowa St,3,2,Y
205,Arizona,Arkansas,1,4,Z
239,Gonzaga,Purdue,3,2,Z


In [22]:
games_16 = add_season_stats(third_round_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, games_16, feature_cols)
result = add_team_names(result, teams)

sweet_16_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR']]
sweet_16_winners = get_round_winners(sweet_16_results)
sweet_16_winners.sort_values('W_REGION')

,W_TEAM_ID,W_TEAM_NAME,W_SEED,W_REGION
0,1181,Duke,1,W
1,1277,Michigan St,3,W
2,1196,Florida,1,X
3,1228,Illinois,3,X
4,1276,Michigan,1,Y
5,1438,Virginia,3,Y
6,1112,Arizona,1,Z
7,1345,Purdue,2,Z


# Sweet 16 winners!

In [23]:
ELITE_8_SEED_PAIRS = [
    (s1, s2)
    for s1 in [1,16,8,9,5,12,4,13]
    for s2 in [6,11,3,14,7,10,2,15]
]

df1 = sweet_16_winners.rename(columns={'W_TEAM_NAME':'W_TEAM_NAME','W_REGION':'W_REGION','W_TEAM_ID':'WTEAMID2','W_SEED':'W_SEED'})
df2 = sweet_16_winners.rename(columns={'W_TEAM_NAME':'L_TEAM_NAME','W_REGION':'L_REGION','W_TEAM_ID':'LTEAMID2','W_SEED':'L_SEED'})

cross = df1.merge(df2, how='cross', suffixes=('','_r'))
seed_set = set(ELITE_8_SEED_PAIRS)
mask = (
    (cross.W_REGION == cross.L_REGION) &
    cross.apply(lambda r: (r.W_SEED, r.L_SEED) in seed_set, axis=1)
)
elite_8_matchups = cross[mask][['W_TEAM_NAME','W_REGION','WTEAMID2','W_SEED','L_TEAM_NAME','L_REGION','LTEAMID2','L_SEED']]
elite_8_matchups.sort_values('W_REGION')

,W_TEAM_NAME,W_REGION,WTEAMID2,W_SEED,L_TEAM_NAME,L_REGION,LTEAMID2,L_SEED
1,Duke,W,1181,1,Michigan St,W,1277,3
19,Florida,X,1196,1,Illinois,X,1228,3
37,Michigan,Y,1276,1,Virginia,Y,1438,3
55,Arizona,Z,1112,1,Purdue,Z,1345,2


# Elite 8

In [24]:
elite_8_matchups[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
).sort_values('REGION')

,TEAM_1,TEAM_2,SEED_1,SEED_2,REGION
1,Duke,Michigan St,1,3,W
19,Florida,Illinois,1,3,X
37,Michigan,Virginia,1,3,Y
55,Arizona,Purdue,1,2,Z


In [25]:
elite_8 = add_season_stats(elite_8_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, elite_8, feature_cols)
result = add_team_names(result, teams)

elite_8_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR']]
elite_8_winners = get_round_winners(elite_8_results)
elite_8_winners.sort_values('W_REGION')

,W_TEAM_ID,W_TEAM_NAME,W_SEED,W_REGION
0,1181,Duke,1,W
1,1196,Florida,1,X
2,1438,Virginia,3,Y
3,1112,Arizona,1,Z


# Elite 8 winners

In [26]:
# Final four matchups: W vs X regions, Y vs Z regions
wx = elite_8_winners[elite_8_winners.W_REGION == 'W'].merge(
    elite_8_winners[elite_8_winners.W_REGION == 'X'].rename(
        columns={'W_TEAM_ID':'TEAM_2','W_TEAM_NAME':'W_TEAM_NAME_2','W_SEED':'SEED_2','W_REGION':'REGION_2'}
    ),
    how='cross',
)

yz = elite_8_winners[elite_8_winners.W_REGION == 'Y'].merge(
    elite_8_winners[elite_8_winners.W_REGION == 'Z'].rename(
        columns={'W_TEAM_ID':'TEAM_2','W_TEAM_NAME':'W_TEAM_NAME_2','W_SEED':'SEED_2','W_REGION':'REGION_2'}
    ),
    how='cross',
)

final_four_matchups = pd.concat([wx, yz]).rename(columns={
    'W_TEAM_ID': 'WTEAMID2',
    'W_TEAM_NAME_2': 'L_TEAM_NAME',
    'REGION_2': 'L_REGION',
    'TEAM_2': 'LTEAMID2',
    'SEED_2': 'L_SEED',
})[['W_TEAM_NAME','W_REGION','WTEAMID2','W_SEED','L_TEAM_NAME','L_REGION','LTEAMID2','L_SEED']]

# FINAL FOUR!

In [27]:
final_four_matchups[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
).sort_values('REGION')

,TEAM_1,TEAM_2,SEED_1,SEED_2,REGION
0,Duke,Florida,1,1,W
0,Virginia,Arizona,3,1,Y


In [28]:
final_four = add_season_stats(final_four_matchups[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, final_four, feature_cols)
result = add_team_names(result, teams)

final_four_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR']]
final_four_winners = get_round_winners(final_four_results)
final_four_winners.sort_values('W_REGION')

,W_TEAM_ID,W_TEAM_NAME,W_SEED,W_REGION
0,1181,Duke,1,W
1,1438,Virginia,3,Y


In [29]:
# Championship: cross-join final four winners and keep the one unique matchup
championship_matchup = (
    final_four_winners.rename(columns={'W_TEAM_ID':'WTEAMID2','W_TEAM_NAME':'W_TEAM_NAME','W_SEED':'W_SEED','W_REGION':'W_REGION'})
    .merge(
        final_four_winners.rename(columns={'W_TEAM_ID':'LTEAMID2','W_TEAM_NAME':'L_TEAM_NAME','W_SEED':'L_SEED','W_REGION':'L_REGION'}),
        how='cross',
    )
    .query('WTEAMID2 != LTEAMID2')
    .head(1)
)

# CHAMPIONSHIP!

In [30]:
championship_matchup[['W_TEAM_NAME','L_TEAM_NAME','W_SEED','L_SEED','W_REGION']].rename(
    columns={'W_TEAM_NAME':'TEAM_1','L_TEAM_NAME':'TEAM_2','W_SEED':'SEED_1','L_SEED':'SEED_2','W_REGION':'REGION'}
)

,TEAM_1,TEAM_2,SEED_1,SEED_2,REGION
1,Duke,Virginia,1,3,W


In [31]:
championship = add_season_stats(championship_matchup[['WTEAMID2', 'LTEAMID2', 'W_SEED', 'L_SEED', 'W_REGION', 'L_REGION']], season_stats, BRACKET_SEASON)
result = predict_games(optimal_model, championship, feature_cols)
result = add_team_names(result, teams)

championship_results = result[['L_TEAMID', 'W_TEAMID', 'L_TEAM_NAME', 'L_SEED', 'W_SEED', 'L_REGION', 'W_TEAM_NAME', 'W_REGION', 'PRED_WIN_INDICATOR']]
championship_results

,L_TEAMID,W_TEAMID,L_TEAM_NAME,L_SEED,W_SEED,L_REGION,W_TEAM_NAME,W_REGION,PRED_WIN_INDICATOR
0,1438,1181,Virginia,3,1,Y,Duke,W,1


In [32]:
champion = get_round_winners(championship_results)
champion

,W_TEAM_ID,W_TEAM_NAME,W_SEED,W_REGION
0,1181,Duke,1,W


# CHAMPION!

## Getting started with 2026 predictions

Before running the bracket prediction cells, you need 2026 tournament seed data in `data_2026/MNCAATourneySeeds.csv`.

Once the seeds are added, re-run `1_cleaning_2026.ipynb` to regenerate `final_season_stats.csv` with 2026 season data, then run this notebook top to bottom.